# `panel_k` (D32) -- chequeos de rutina

Objetivo: validar que `panel_k` hace lo que se diseñó para hacer, con los 3 niveles, antes de
usarlo para nada analítico. Exploratorio -- cada celda de código verifica algo puntual con datos
reales; la conclusión al final solo resume lo que las celdas de arriba mostraron.

### Paso 1 -- Cargar panel_k, modo "nivel" y "delta", los 3 niveles, K=4

In [1]:
import sys
general_path = "/workspaces/analisis-politica-economia/"
sys.path.insert(0, f"{general_path}src")

import random
import pandas as pd

from ml_models.panel_k import construir_panel_k, columnas_candidatas_k, COLUMNAS_METADATA_PANEL_K
from ml_models.cargar_series_economicas import cargar_registro

PANEL_TRIMESTRAL_DIR = f"{general_path}data/tfi_data/panel/t-1"
REGISTRO_VARIABLES_PATH = f"{general_path}data/tfi_data/registro_variables.csv"

NIVELES = ("municipal", "provincial", "nacional")

def _panel_k(nivel, **kwargs):
    return construir_panel_k(nivel, panel_dir=PANEL_TRIMESTRAL_DIR, registro_path=REGISTRO_VARIABLES_PATH, **kwargs)

PREFIJOS_EPH = (
    "tasa_informalidad", "pct_sin_cobertura_salud", "hacinamiento_medio",
    "pct_hogares_ayuda_social_gobierno", "pct_hogares_prestamo_bancario",
    "pct_hogares_vendio_pertenencias",
)

for nivel in NIVELES:
    for modo in ("nivel", "delta"):
        df = _panel_k(nivel, k=4, modo=modo)
        print(f"{nivel:12s} modo={modo:6s} shape={df.shape}")

municipal    modo=nivel  shape=(12, 82)
municipal    modo=delta  shape=(11, 147)
provincial   modo=nivel  shape=(12, 82)


provincial   modo=delta  shape=(11, 147)
nacional     modo=nivel  shape=(12, 82)


nacional     modo=delta  shape=(11, 147)


### Paso 2 -- Matriz de cobertura: filas por (nivel, modo, K) y cuántas tienen las 6 EPH sin `NaN` en `_nivel_kt`

Esto tiene que mostrar la ganancia que motivó `panel_k`: en modo "nivel", `*_2003_2005` entra con
EPH completa; en modo "delta" entra con `kt` completo y `kt1`/`kd` en `NaN` para EPH (documentado,
no oculto); `*_2001_2003` no entra en ningún modo/K -- confirmado abajo con datos reales, no solo
descripto.

In [2]:
filas = []
for nivel in NIVELES:
    for modo in ("nivel", "delta"):
        for k in range(1, 9):
            df = _panel_k(nivel, k=k, modo=modo)
            cols_eph_kt = [f"{p}_nivel_kt" for p in PREFIJOS_EPH]
            eph_completa = df[cols_eph_kt].notna().all(axis=1) if len(df) else pd.Series([], dtype=bool)
            filas.append({
                "nivel": nivel, "modo": modo, "k": k,
                "n_filas": len(df),
                "n_con_eph_kt_completa": int(eph_completa.sum()),
            })
matriz_cobertura = pd.DataFrame(filas)
matriz_cobertura

,nivel,modo,k,n_filas,n_con_eph_kt_completa
0,municipal,nivel,1,12,10
1,municipal,nivel,2,12,11
2,municipal,nivel,3,12,11
3,municipal,nivel,4,12,11
4,municipal,nivel,5,12,11
5,municipal,nivel,6,12,11
6,municipal,nivel,7,12,11
7,municipal,nivel,8,12,11
8,municipal,delta,1,11,10
9,municipal,delta,2,11,11


In [3]:
# Confirmación explícita: 2003_2005 vs 2001_2003, para K=1..8, nacional (idéntico en los otros
# 2 niveles -- misma serie EPH/macro, verificado en el plan de panel_k).
for k in range(1, 9):
    df_nivel = _panel_k("nacional", k=k, modo="nivel")
    df_delta = _panel_k("nacional", k=k, modo="delta")

    fila_2005_nivel = df_nivel[df_nivel["id_transicion"] == "nacional_2003_2005"]
    fila_2003_nivel = df_nivel[df_nivel["id_transicion"] == "nacional_2001_2003"]
    fila_2005_delta = df_delta[df_delta["id_transicion"] == "nacional_2003_2005"]

    eph_kt_2005_nivel = fila_2005_nivel[[f"{p}_nivel_kt" for p in PREFIJOS_EPH]].iloc[0].tolist() if len(fila_2005_nivel) else None
    eph_kt_2003_nivel = fila_2003_nivel[[f"{p}_nivel_kt" for p in PREFIJOS_EPH]].iloc[0].tolist() if len(fila_2003_nivel) else None
    print(f"K={k}")
    print(f"  nacional_2003_2005, modo nivel, _nivel_kt EPH (6 vars): {eph_kt_2005_nivel}")
    print(f"  nacional_2001_2003, modo nivel, _nivel_kt EPH (6 vars): {eph_kt_2003_nivel}")
    if len(fila_2005_delta):
        kt = fila_2005_delta[[f"{p}_nivel_kt" for p in PREFIJOS_EPH]].iloc[0].tolist()
        kt1 = fila_2005_delta[[f"{p}_nivel_kt1" for p in PREFIJOS_EPH]].iloc[0].tolist()
        kd = fila_2005_delta[[f"{p}_nivel_kd" for p in PREFIJOS_EPH]].iloc[0].tolist()
        print(f"  nacional_2003_2005, modo delta, _nivel_kt  EPH (6 vars): {kt}")
        print(f"  nacional_2003_2005, modo delta, _nivel_kt1 EPH (6 vars): {kt1}")
        print(f"  nacional_2003_2005, modo delta, _nivel_kd  EPH (6 vars): {kd}")
    else:
        print("  nacional_2003_2005, modo delta: fila ausente para este K")
    print()

K=1
  nacional_2003_2005, modo nivel, _nivel_kt EPH (6 vars): [0.4122248714606192, 0.35543438561397, 1.105174646281725, 0.0059296163979027, 0.1275582386918199, 0.0958354193581889]
  nacional_2001_2003, modo nivel, _nivel_kt EPH (6 vars): [nan, nan, nan, nan, nan, nan]
  nacional_2003_2005, modo delta, _nivel_kt  EPH (6 vars): [0.4122248714606192, 0.35543438561397, 1.105174646281725, 0.0059296163979027, 0.1275582386918199, 0.0958354193581889]
  nacional_2003_2005, modo delta, _nivel_kt1 EPH (6 vars): [nan, nan, nan, nan, nan, nan]
  nacional_2003_2005, modo delta, _nivel_kd  EPH (6 vars): [nan, nan, nan, nan, nan, nan]

K=2
  nacional_2003_2005, modo nivel, _nivel_kt EPH (6 vars): [0.41075070980225714, 0.35160494879515236, 1.0958431962362902, 0.0057406397116009, 0.1316319308706704, 0.09862364720549274]
  nacional_2001_2003, modo nivel, _nivel_kt EPH (6 vars): [nan, nan, nan, nan, nan, nan]
  nacional_2003_2005, modo delta, _nivel_kt  EPH (6 vars): [0.41075070980225714, 0.351604948795152

K=3
  nacional_2003_2005, modo nivel, _nivel_kt EPH (6 vars): [0.4164093687985877, 0.3533035927297861, 1.098442762555665, 0.004980223638066233, 0.12929531927524773, 0.09948955975237457]
  nacional_2001_2003, modo nivel, _nivel_kt EPH (6 vars): [nan, nan, nan, nan, nan, nan]
  nacional_2003_2005, modo delta, _nivel_kt  EPH (6 vars): [0.4164093687985877, 0.3533035927297861, 1.098442762555665, 0.004980223638066233, 0.12929531927524773, 0.09948955975237457]
  nacional_2003_2005, modo delta, _nivel_kt1 EPH (6 vars): [nan, nan, nan, nan, nan, nan]
  nacional_2003_2005, modo delta, _nivel_kd  EPH (6 vars): [nan, nan, nan, nan, nan, nan]



K=4
  nacional_2003_2005, modo nivel, _nivel_kt EPH (6 vars): [0.41904223038732014, 0.3562670958944131, 1.1194197692919834, 0.005730576456522725, 0.12021436570124941, 0.09713580174107407]
  nacional_2001_2003, modo nivel, _nivel_kt EPH (6 vars): [nan, nan, nan, nan, nan, nan]
  nacional_2003_2005, modo delta, _nivel_kt  EPH (6 vars): [0.41904223038732014, 0.3562670958944131, 1.1194197692919834, 0.005730576456522725, 0.12021436570124941, 0.09713580174107407]
  nacional_2003_2005, modo delta, _nivel_kt1 EPH (6 vars): [nan, nan, nan, nan, nan, nan]
  nacional_2003_2005, modo delta, _nivel_kd  EPH (6 vars): [nan, nan, nan, nan, nan, nan]

K=5
  nacional_2003_2005, modo nivel, _nivel_kt EPH (6 vars): [0.41629089089221216, 0.3555028982764116, 1.1386100183697938, 0.00667397918504864, 0.1124533392491072, 0.09316865941704984]
  nacional_2001_2003, modo nivel, _nivel_kt EPH (6 vars): [nan, nan, nan, nan, nan, nan]
  nacional_2003_2005, modo delta, _nivel_kt  EPH (6 vars): [0.41629089089221216, 0

K=6
  nacional_2003_2005, modo nivel, _nivel_kt EPH (6 vars): [0.4138785766732229, 0.3525010310049443, 1.1487175797351472, 0.00749359217541815, 0.10745343408911288, 0.09493806294460898]
  nacional_2001_2003, modo nivel, _nivel_kt EPH (6 vars): [nan, nan, nan, nan, nan, nan]
  nacional_2003_2005, modo delta, _nivel_kt  EPH (6 vars): [0.4138785766732229, 0.3525010310049443, 1.1487175797351472, 0.00749359217541815, 0.10745343408911288, 0.09493806294460898]
  nacional_2003_2005, modo delta, _nivel_kt1 EPH (6 vars): [nan, nan, nan, nan, nan, nan]
  nacional_2003_2005, modo delta, _nivel_kd  EPH (6 vars): [nan, nan, nan, nan, nan, nan]

K=7
  nacional_2003_2005, modo nivel, _nivel_kt EPH (6 vars): [0.41108965501320077, 0.35258777235686917, 1.1575635998816562, 0.007667986154324429, 0.10024228555647714, 0.10123307641461476]
  nacional_2001_2003, modo nivel, _nivel_kt EPH (6 vars): [nan, nan, nan, nan, nan, nan]
  nacional_2003_2005, modo delta, _nivel_kt  EPH (6 vars): [0.41108965501320077, 0.

K=8
  nacional_2003_2005, modo nivel, _nivel_kt EPH (6 vars): [0.41255681381941434, 0.35649076391133716, 1.1656649337456613, 0.007726446067551938, 0.0928516046742281, 0.1022302410146546]
  nacional_2001_2003, modo nivel, _nivel_kt EPH (6 vars): [nan, nan, nan, nan, nan, nan]
  nacional_2003_2005, modo delta, _nivel_kt  EPH (6 vars): [0.41255681381941434, 0.35649076391133716, 1.1656649337456613, 0.007726446067551938, 0.0928516046742281, 0.1022302410146546]
  nacional_2003_2005, modo delta, _nivel_kt1 EPH (6 vars): [nan, nan, nan, nan, nan, nan]
  nacional_2003_2005, modo delta, _nivel_kd  EPH (6 vars): [nan, nan, nan, nan, nan, nan]



### Paso 3 -- `trimestre_contaminado_ipc()` como guarda general (D33 §5)

No existía como código todavía (solo descripta en la especificación) -- se define acá, mínima,
como guarda de rutina: marca cualquier trimestre con `ipc` (variación % trimestral, D13) por
debajo de un umbral de deflación implausible. El umbral (-10%) se fija mirando el rango real de
la serie ya arreglada (D33): hoy va de -2.12% a +70.72% (este último compatible con la crisis
2001-2002, no un artefacto) -- el salto roto que motivó D33 llegaba a -39.11%, muy por debajo de
cualquier trimestre real de la serie.

In [4]:
UMBRAL_DEFLACION_IMPLAUSIBLE = -10.0

def trimestre_contaminado_ipc(valor, umbral=UMBRAL_DEFLACION_IMPLAUSIBLE):
    return valor is not None and pd.notna(valor) and valor < umbral

contaminados = []
for nivel in NIVELES:
    panel = pd.read_csv(f"{PANEL_TRIMESTRAL_DIR}/panel_trimestral_{nivel}.csv")
    tri = panel[panel["tipo_fila"] == "trimestre"]
    marcados = tri[tri["ipc"].apply(trimestre_contaminado_ipc)]
    for _, fila in marcados.iterrows():
        contaminados.append({"nivel": nivel, "id_transicion": fila["id_transicion"], "fecha_inicio": fila["fecha_inicio"], "ipc": fila["ipc"]})
    print(f"{nivel}: ipc trimestral min={tri['ipc'].min():.4f} max={tri['ipc'].max():.4f} -- {len(marcados)} trimestre(s) contaminado(s)")

print()
if contaminados:
    print("Trimestres marcados (mostrados explícitamente, no asumido vacío):")
    display(pd.DataFrame(contaminados))
else:
    print("Ningún trimestre marcado en los 3 niveles -- confirmado explícitamente, no asumido.")

municipal: ipc trimestral min=-2.1244 max=70.7158 -- 0 trimestre(s) contaminado(s)
provincial: ipc trimestral min=-2.1244 max=70.7158 -- 0 trimestre(s) contaminado(s)
nacional: ipc trimestral min=-2.1244 max=70.7158 -- 0 trimestre(s) contaminado(s)

Ningún trimestre marcado en los 3 niveles -- confirmado explícitamente, no asumido.


### Paso 4 -- Validación de integridad interna: `nivel_kt` vs. promedio simple de esos K trimestres

Reemplaza la comparación contra `panel_ventanas.csv` (no aplica -- `panel_k` no se ancla en
transiciones vc/vl). 3 combinaciones (nivel, variable, id_transicion, K) elegidas al azar
(semilla fija para reproducibilidad) entre las que efectivamente tienen dato real.

In [5]:
random.seed(20260932)

registro = cargar_registro(REGISTRO_VARIABLES_PATH)
variables_completas = [v.id_variable for v in registro if v.paquete_atributos == "completo"]

combos_candidatos = []
for nivel in NIVELES:
    panel = pd.read_csv(f"{PANEL_TRIMESTRAL_DIR}/panel_trimestral_{nivel}.csv")
    for id_transicion, grupo in panel.groupby("id_transicion"):
        tri = grupo[grupo["tipo_fila"] == "trimestre"].sort_values("orden")
        for var in variables_completas:
            if tri[var].notna().sum() >= 4:
                combos_candidatos.append((nivel, var, id_transicion, tri))

muestra = random.sample(combos_candidatos, 3)

for nivel, var, id_transicion, tri in muestra:
    k = random.choice([2, 4, 6, 8])
    ultimos_k = tri.tail(k)
    promedio_manual = ultimos_k[var].mean()

    df_k = _panel_k(nivel, k=k, modo="nivel")
    fila = df_k[df_k["id_transicion"] == id_transicion]
    nivel_kt = fila[f"{var}_nivel_kt"].iloc[0] if len(fila) else None

    print(f"nivel={nivel} variable={var!r} id_transicion={id_transicion!r} K={k}")
    print(f"  valores usados (últimos {k} trimestres): {ultimos_k[var].tolist()}")
    print(f"  promedio simple manual : {promedio_manual}")
    print(f"  {var}_nivel_kt (panel_k): {nivel_kt}")
    print(f"  coincide (abs diff < 1e-9): {abs(promedio_manual - nivel_kt) < 1e-9 if nivel_kt is not None else 'N/A -- panel_k dio None'}")
    print()

nivel=provincial variable='reservas' id_transicion='provincial_2009_2011' K=6


  valores usados (últimos 6 trimestres): [49382.71831998723, 50912.218592640886, 52244.43941269314, 52117.14416282642, 51977.76666666666, 49282.27956989248]
  promedio simple manual : 50986.09445411781
  reservas_nivel_kt (panel_k): 50986.094454117796
  coincide (abs diff < 1e-9): True

nivel=provincial variable='icg' id_transicion='provincial_2021_2023' K=8
  valores usados (últimos 8 trimestres): [1.4861313899358113, 1.4139509201049805, 1.2336197296778362, 1.2387876907984416, 1.2322392463684082, 1.119141658147176, 1.198828895886739, 1.1202370524406433]
  promedio simple manual : 1.2553670729200044
  icg_nivel_kt (panel_k): 1.2553670729200046
  coincide (abs diff < 1e-9): True

nivel=municipal variable='icg' id_transicion='municipal_2007_2009' K=6
  valores usados (últimos 6 trimestres): [1.8519647518793745, 1.267168362935384, 1.4052430788675945, 1.4333923657735188, 1.3622303406397502, 1.3309803009033203]
  promedio simple manual : 1.4418298668331573
  icg_nivel_kt (panel_k): 1.44182

### Paso 5 -- `columnas_candidatas_k`: modo "nivel" vs. "delta", K=4, los 3 niveles

In [6]:
filas = []
for nivel in NIVELES:
    for modo in ("nivel", "delta"):
        df = _panel_k(nivel, k=4, modo=modo)
        cols = columnas_candidatas_k(df)
        filas.append({"nivel": nivel, "modo": modo, "n_columnas_candidatas": len(cols)})
pd.DataFrame(filas)

,nivel,modo,n_columnas_candidatas
0,municipal,nivel,61
1,municipal,delta,122
2,provincial,nivel,61
3,provincial,delta,122
4,nacional,nivel,61
5,nacional,delta,122


### Conclusión

Todo lo de abajo describe únicamente lo que mostraron las celdas de arriba, en esta corrida.

- **Paso 1**: `panel_k` carga sin error en los 3 niveles y los 2 modos. Modo "nivel" da 12 filas
  (una por cada una de las 12 transiciones de `panel_trimestral_<nivel>.csv`); modo "delta" da 11
  (se saltea `*_2001_2003`, sin transición anterior, como estaba previsto). Mismo shape en los 3
  niveles (82 columnas en "nivel", 147 en "delta" -- la fuente EPH/macro es la misma serie
  nacional para los 3, solo cambian las fronteras electorales).
- **Paso 2**: para `K=2..8`, 11 de 12 filas (modo "nivel") y 11 de 11 (modo "delta") tienen las 6
  EPH sin `NaN` en `_nivel_kt`, en los 3 niveles -- la única fila sin EPH es `*_2001_2003` (que en
  modo "delta" ni siquiera llega a existir como fila). Para `K=1` baja a 10/12 y 10/11: además de
  `*_2001_2003`, se suma `*_2013_2015` -- confirmado por separado (no en la celda de arriba) que
  su único trimestre a `K=1` (2015-08/10, el más cercano a la elección de oct-2015) cae justo en
  el hueco real de EPH 2015T3/2015T4 ya documentado en `docs/FUNCIONALIDADES.md`, no un bug de
  `panel_k`. Con `K>=2` esa fila se recupera porque el trimestre anterior (2015-05/07) sí tiene
  dato.
- **Paso 2 (celda de confirmación explícita)**: para los 8 valores de `K`, `nacional_2003_2005`
  tiene las 6 `_nivel_kt` de EPH con valor real tanto en modo "nivel" como en modo "delta";
  `nacional_2001_2003` da `NaN` en las 6, para los 8 `K`, en modo "nivel"; en modo "delta",
  `nacional_2003_2005` tiene `_nivel_kt1`/`_nivel_kd` en `NaN` para las 6 EPH, en los 8 `K` --
  exactamente la ganancia de diseño que motivó `panel_k` (D32), confirmada con datos reales, no
  solo descripta.
- **Paso 3**: `trimestre_contaminado_ipc()` (umbral -10% de variación trimestral) no marcó ningún
  trimestre en `panel_trimestral_<nivel>.csv`, en los 3 niveles -- rango real hoy: -2.12% a
  +70.72% (compatible con la crisis 2001-2002). Confirma, con un chequeo independiente del de
  D33, que el fix de IPC/FACPCE sigue sano en la fuente que consume `panel_k`.
- **Paso 4**: las 3 combinaciones (nivel, variable, transición, K) elegidas al azar
  (`reservas`/`provincial_2009_2011`/K=6, `icg`/`provincial_2021_2023`/K=8,
  `icg`/`municipal_2007_2009`/K=6) dieron `nivel_kt` idéntico (diferencia < 1e-9) al promedio
  simple de esos mismos trimestres tomados directo de `panel_trimestral_<nivel>.csv`.
- **Paso 5**: `columnas_candidatas_k` da 61 columnas en modo "nivel" y 122 en modo "delta"
  (~2x, ya que "delta" duplica cada atributo en `kt`/`kt1` más `nivel_kd`), igual en los 3
  niveles.
